# 05 · GeoPandas: นำเข้าตำแหน่งสถานีวัดน้ำฝน + ขอบเขตการปกครอง แล้วทำแผนที่
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/05_Stations_and_Boundaries_GeoPandas.ipynb)

**เวลา:** 3 ชั่วโมง | **ระดับ:** 🟡 ปานกลาง → 🔴 ยาก

```
ตาราง CSV (lat, lon)  ──►  GeoDataFrame (จุด)  ─┐
                                                 ├─► Spatial Join / Clip / Buffer ─► แผนที่
ไฟล์ขอบเขต (.shp/.geojson) ──► GeoDataFrame (พื้นที่) ─┘
```

**GeoDataFrame** = DataFrame ของ Pandas + คอลัมน์ `geometry` + ระบบพิกัด (CRS)

| ระบบพิกัด | EPSG | หน่วย | ใช้เมื่อ |
|---|---|---|---|
| WGS 84 (ภูมิศาสตร์) | 4326 | องศา | เก็บ/แสดงผลทั่วไป, GPS |
| WGS 84 / UTM zone 47N | 32647 | เมตร | วัดระยะ พื้นที่ buffer (ประเทศไทยส่วนใหญ่) |
| WGS 84 / UTM zone 48N | 32648 | เมตร | ภาคตะวันออกเฉียงเหนือฝั่งตะวันออก |

> ⚠️ **ห้ามวัดระยะทาง/พื้นที่ในระบบ EPSG:4326** เพราะหน่วยเป็นองศา ต้อง `to_crs()` เป็น UTM ก่อนเสมอ

In [ ]:
# ============================================================
# ตั้งค่าฟอนต์ภาษาไทยให้ Matplotlib (ถ้าไม่ตั้ง ตัวอักษรไทยจะเป็นสี่เหลี่ยม □□□)
# ============================================================
import glob, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def setup_thai_font():
    fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    if not fonts:   # บน Colab ต้องติดตั้งก่อน (ใช้เวลา ~10 วินาที ครั้งเดียวต่อ session)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null", shell=True)
        fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    for f in fonts:
        fm.fontManager.addfont(f)
    if fonts:
        plt.rcParams["font.family"] = "Loma"
        print("✅ ใช้ฟอนต์ไทย Loma")
    else:
        print("⚠️ ติดตั้งฟอนต์ไทยไม่ได้ ใช้ฟอนต์เริ่มต้นแทน")
    plt.rcParams["axes.unicode_minus"] = False   # ให้เครื่องหมายลบแสดงถูกต้อง

setup_thai_font()

In [ ]:
# ============================================================
# เซลล์โหลดข้อมูล (ใช้ซ้ำทุก Notebook)
# ลำดับการหา: 1) โฟลเดอร์ data/ ในเครื่อง  2) GitHub ของอาจารย์
# ============================================================
import os
import pandas as pd

GITHUB_USER = "apichayaice7835-png"      # ← ชื่อบัญชี GitHub ของอาจารย์
REPO        = "Teach-Python-Geography"   # ← ชื่อ repository
BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def data_path(filename):
    """คืน path ของไฟล์: ถ้ามีในเครื่องใช้ในเครื่อง ถ้าไม่มีใช้ลิงก์ GitHub"""
    for folder in ["data", f"{REPO}/data", "/content/data"]:
        p = os.path.join(folder, filename)
        if os.path.exists(p):
            return p
    return BASE_URL + filename

print("ไฟล์สถานี :", data_path("rain_stations_north.csv"))
print("ไฟล์ฝนรายเดือน:", data_path("rain_monthly_2020_2024.csv"))

In [ ]:
import numpy as np
import geopandas as gpd
print("GeoPandas", gpd.__version__)

## 🟡 1. แปลงตาราง CSV เป็นจุด (Points)

In [ ]:
stations = pd.read_csv(data_path("rain_stations_north.csv"))

gdf_st = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["lon"], stations["lat"]),   # ⚠️ ลำดับคือ (x=ลองจิจูด, y=ละติจูด)
    crs="EPSG:4326",                                                   # บอกว่าพิกัดเป็นองศา WGS84
)
print(gdf_st.crs)
gdf_st.head()

In [ ]:
gdf_st.plot(marker="^", color="blue", markersize=40, figsize=(5, 6))
plt.title("ตำแหน่งสถานี (ยังไม่มีขอบเขต)"); plt.show()

## 🟡 2. นำเข้าขอบเขตจังหวัด (Polygon)
เลือกได้ 3 แหล่ง — **แนะนำให้ดาวน์โหลดไฟล์ขอบเขตมาเก็บไว้ใน `data/` ของ repo ตนเอง** เพื่อไม่ให้แบบเรียนพังเมื่อลิงก์ภายนอกเปลี่ยน

In [ ]:
# ---------- แหล่งที่ 1: ไฟล์ในโฟลเดอร์ data/ ของ repo (ถ้ามี) ----------
# ---------- แหล่งที่ 2: GeoJSON ขอบเขตจังหวัดจาก GitHub สาธารณะ ----------
PROV_URL = "https://raw.githubusercontent.com/apisit/thailand.json/master/thailand.json"

local = data_path("thailand_provinces.geojson")
src = local if os.path.exists(local) else PROV_URL
provinces = gpd.read_file(src)
print("อ่านจาก:", src)
print(provinces.shape, provinces.crs)
provinces.head()

In [ ]:
# ---------- แหล่งที่ 3: ไฟล์ของตนเอง (Shapefile บีบอัด .zip / GeoPackage / GeoJSON) ----------
# from google.colab import files
# up = files.upload()                                   # เลือกไฟล์ เช่น amphoe.zip
# my_boundary = gpd.read_file(list(up.keys())[0])       # GeoPandas อ่าน .zip ของ Shapefile ได้โดยตรง
# print(my_boundary.crs)                                # ⚠️ ตรวจ CRS ทุกครั้ง
# my_boundary = my_boundary.to_crs("EPSG:4326")         # แปลงให้ตรงกับชั้นข้อมูลอื่น
#
# Shapefile ภาษาไทยอ่านแล้วเพี้ยน? ลอง: gpd.read_file("x.shp", encoding="cp874")

In [ ]:
# เลือกเฉพาะ 9 จังหวัดภาคเหนือตอนบน + อุตรดิตถ์ (ตามที่ตั้งสถานี)
NORTH = ["Chiang Mai", "Chiang Rai", "Lampang", "Lamphun", "Mae Hong Son",
         "Nan", "Phayao", "Phrae", "Uttaradit"]
north = provinces[provinces["name"].isin(NORTH)].copy()
north = north.rename(columns={"name": "prov_name"})
print(len(north), "จังหวัด")

fig, ax = plt.subplots(figsize=(7, 8))
provinces.plot(ax=ax, color="whitesmoke", edgecolor="lightgray")
north.plot(ax=ax, color="lightyellow", edgecolor="black")
gdf_st.plot(ax=ax, marker="^", color="blue", markersize=40, label="สถานีวัดน้ำฝน")
ax.set_xlim(97.2, 101.6); ax.set_ylim(17.0, 20.7)
ax.legend(); ax.set_title("สถานีวัดน้ำฝน และขอบเขตจังหวัดภาคเหนือ")
plt.show()

## 🟡 3. Spatial Join: สถานีแต่ละแห่งอยู่ในจังหวัดใด
ใช้ตรวจสอบคุณภาพข้อมูล — คอลัมน์ `province` ที่พิมพ์ไว้ ตรงกับตำแหน่งจริงหรือไม่

In [ ]:
joined = gpd.sjoin(gdf_st, north[["prov_name", "geometry"]], how="left", predicate="within")
joined["check"] = np.where(joined["province"] == joined["prov_name"], "✅ ตรง", "❌ ไม่ตรง")
joined[["station_id", "name_th", "province", "prov_name", "check"]]

In [ ]:
# นับจำนวนสถานีต่อจังหวัด และความหนาแน่นสถานี (ต้องใช้ UTM เพื่อคำนวณพื้นที่)
north_utm = north.to_crs(epsg=32647)
north_utm["area_km2"] = north_utm.area / 1e6
count = joined.groupby("prov_name").size().rename("n_station")
north_utm = north_utm.merge(count, on="prov_name", how="left").fillna({"n_station": 0})
north_utm["km2_per_station"] = (north_utm["area_km2"] / north_utm["n_station"]).round(0)
north_utm[["prov_name", "area_km2", "n_station", "km2_per_station"]].sort_values("km2_per_station")

## 🟡 4. Buffer และระยะทาง (ใน UTM)
คำถาม: *พื้นที่ภาคเหนือกี่ % ที่อยู่ห่างจากสถานีวัดฝนไม่เกิน 20 กม.?*

In [ ]:
st_utm = gdf_st.to_crs(epsg=32647)
buf20 = st_utm.buffer(20_000)                          # รัศมี 20,000 เมตร
region = north_utm.dissolve()                          # รวมทุกจังหวัดเป็นพื้นที่เดียว
covered = buf20.union_all().intersection(region.geometry.iloc[0])
pct = covered.area / region.area.iloc[0] * 100
print(f"พื้นที่ที่อยู่ภายใน 20 กม. จากสถานี = {pct:.1f}%")

fig, ax = plt.subplots(figsize=(7, 8))
north_utm.plot(ax=ax, color="lightyellow", edgecolor="gray")
gpd.GeoSeries([covered], crs=32647).plot(ax=ax, color="skyblue", alpha=0.6)
st_utm.plot(ax=ax, color="blue", marker="^", markersize=30)
ax.set_title(f"พื้นที่ครอบคลุมรัศมี 20 กม. ({pct:.1f}%)"); ax.set_axis_off(); plt.show()

In [ ]:
# ตารางระยะทางระหว่างสถานี (distance matrix) หน่วย กม.
dist = pd.DataFrame(
    {n: (st_utm.distance(g) / 1000).values for n, g in zip(st_utm["name_th"], st_utm.geometry)},
    index=st_utm["name_th"]).round(0)
dist.iloc[:6, :6]

## 🔴 5. แผนที่ Proportional Symbol + Choropleth
เชื่อมผลวิเคราะห์ฝนจาก Notebook 03 เข้ากับชั้นข้อมูลเชิงพื้นที่

In [ ]:
rain = pd.read_csv(data_path("rain_monthly_2020_2024.csv"), parse_dates=["date"])
rain["year"] = rain["date"].dt.year
annual = (rain.groupby(["station_id", "year"])["rain_mm"].sum()
              .groupby("station_id").mean().rename("annual_mm").reset_index())
gdf_st = gdf_st.merge(annual, on="station_id")

# ค่าเฉลี่ยฝนรายปีระดับจังหวัด (ค่าเฉลี่ยเลขคณิตของสถานีในจังหวัด)
prov_rain = (gpd.sjoin(gdf_st, north[["prov_name", "geometry"]], predicate="within")
                .groupby("prov_name")["annual_mm"].mean().rename("prov_annual_mm"))
north = north.merge(prov_rain, on="prov_name", how="left")

fig, ax = plt.subplots(figsize=(8, 9))
north.plot(ax=ax, column="prov_annual_mm", cmap="Blues", edgecolor="black",
           legend=True, legend_kwds={"label": "ฝนรายปีเฉลี่ยระดับจังหวัด (มม.)", "shrink": 0.6})
gdf_st.plot(ax=ax, markersize=(gdf_st["annual_mm"] / 12), color="orange",
            edgecolor="k", alpha=0.8)                              # ขนาดจุดแปรตามปริมาณฝน
for _, r in gdf_st.iterrows():
    ax.annotate(f"{r['name_th']}\n{r['annual_mm']:.0f}", (r.geometry.x, r.geometry.y),
                fontsize=7, ha="center", xytext=(0, 9), textcoords="offset points")
for _, r in north.iterrows():                                      # ชื่อจังหวัดที่จุดศูนย์กลาง
    p = r.geometry.representative_point()
    ax.text(p.x, p.y - 0.12, r["prov_name"], fontsize=8, ha="center", color="darkblue", style="italic")

# องค์ประกอบแผนที่: ลูกศรทิศเหนือ + แหล่งข้อมูล
ax.annotate("N", xy=(0.93, 0.95), xytext=(0.93, 0.87), xycoords="axes fraction",
            ha="center", fontsize=14, arrowprops=dict(facecolor="black", width=4, headwidth=12))
ax.text(0.01, 0.01, "ข้อมูล: ข้อมูลจำลองเพื่อการสอน | ขอบเขต: thailand.json", transform=ax.transAxes, fontsize=7)
ax.set_title("ปริมาณฝนรายปีเฉลี่ย 2020–2024 ภาคเหนือ", fontsize=14)
ax.set_xlabel("ลองจิจูด"); ax.set_ylabel("ละติจูด")
os.makedirs("output", exist_ok=True)
fig.savefig("output/map_annual_rain_north.png", dpi=300, bbox_inches="tight")
plt.show()

## 🔴 6. ส่งออกไฟล์เพื่อเปิดใน QGIS / ArcGIS

In [ ]:
gdf_st.to_file("output/rain_stations_north.gpkg", layer="stations", driver="GPKG")     # GeoPackage (แนะนำ)
north.to_file("output/rain_stations_north.gpkg", layer="provinces", driver="GPKG")
gdf_st.to_file("output/rain_stations_north.geojson", driver="GeoJSON")
north.to_file("output/north_provinces.geojson", driver="GeoJSON")     # เก็บไว้ใน data/ ของ repo ได้
print(os.listdir("output"))
# ดาวน์โหลดลงเครื่อง (Colab):  from google.colab import files; files.download("output/rain_stations_north.gpkg")

## 📎 Checklist เมื่อนำเข้าไฟล์ของตนเอง
| ปัญหา | วิธีแก้ |
|---|---|
| CSV มีพิกัดเป็น UTM (เมตร) ไม่ใช่องศา | `gpd.points_from_xy(df.E, df.N)` แล้ว `crs="EPSG:32647"` |
| พิกัดเป็น DMS เช่น `18°47'24"` | แปลงด้วยฟังก์ชัน `dms_to_dd` จาก Notebook 01 |
| จุดไปโผล่ทะเล/ต่างประเทศ | สลับ lat/lon กัน → ตรวจว่า x = lon, y = lat |
| Shapefile ไม่มี `.prj` | `gdf.set_crs(epsg=...)` (**set** ไม่ใช่ **to**) |
| ชั้นข้อมูลไม่ซ้อนกัน | CRS ไม่ตรงกัน → `to_crs()` ให้เหมือนกัน |
| ชื่อจังหวัดสะกดต่างกัน (Lop Buri / Lopburi) | ใช้รหัสจังหวัดในการ join แทนชื่อ |

---
## ✏️ แบบฝึกหัด
**🟢** 1) เปลี่ยนสัญลักษณ์และสีจุดสถานี  2) แสดงเฉพาะสถานีที่สูงกว่า 300 ม.

**🟡** 3) หาสถานีที่ใกล้ "ตัวเมืองน่าน" ที่สุด 3 แห่ง (`distance` ใน UTM)
4) เปลี่ยนรัศมี buffer เป็น 10, 20, 30 กม. แล้ววาดกราฟ % พื้นที่ครอบคลุม

**🔴** 5) นำเข้า Shapefile ขอบเขต **อำเภอ** ของตนเอง แล้วนับจำนวนสถานีต่ออำเภอ
6) หา "ช่องว่างของเครือข่ายสถานี": จุดใดในภาคเหนือไกลจากสถานีที่ใกล้ที่สุดมากที่สุด (hint: สร้างกริดจุด แล้วใช้ `sjoin_nearest`)